<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/03_Tab_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =================================================================
# [Bass Separator] Integrated Setup & Initialization
# =================================================================
from google.colab import drive
import os
import sys

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 최신화 (Git Clone / Pull)
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

if not os.path.exists(PROJECT_PATH):
    print(f"📦 Cloning repository... ({PROJECT_NAME})")
    !git clone {REPO_URL}
else:
    print(f"🔄 Updating repository... (Git Pull)")
    !cd {PROJECT_PATH} && git pull

# 3. 작업 경로 설정
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f"📂 Working Directory: {os.getcwd()}")

# 4. [src] 모듈을 이용한 환경 구축 및 데이터 로드
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # A. 설치 및 환경 설정 (FFmpeg, Demucs, 호환성 해결)
    init_colab_env()

    # B. 데이터셋 로드
    MY_DRIVE_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_PATH)

except ImportError as e:
    print(f"⚠️ src 모듈 로드 실패: {e}")
    print("   Git Clone이 정상적으로 되었는지 확인해주세요.")

# 5. 자주 쓰는 라이브러리 임포트 (편의용)
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import soundfile as sf
from IPython.display import Audio, display
import subprocess
print("📚 Standard Libraries Imported.")

In [ ]:
# 분석할 오디오 파일의 경로를 입력
target_file_path = "/content/drive/MyDrive/Bass_separator/dataset/기타 베이스 분리 예제 2.wav"

# Demucs로 베이스 트랙 분리
print("🚀 Demucs 분리 시작...")

# 파일명 추출 (확장자 제외)
filename = os.path.splitext(os.path.basename(target_file_path))[0]

# Demucs 실행 (htdemucs 모델 사용)
cmd = f'demucs -n htdemucs "{target_file_path}"'
process = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# 분리된 베이스 파일 경로 자동 탐색
# Demucs 기본 출력 경로: separated/htdemucs/파일명/bass.wav
bass_stem_path = os.path.join('separated', 'htdemucs', filename, 'bass.wav')
print(f"✅ 분리 완료! 베이스 트랙 경로: {bass_stem_path}")

# 오디오 재생
print("🎧 분리된 베이스 트랙 듣기:")
display(Audio(bass_stem_path))

In [ ]:
from src.bass_transcription import detect_pitch

 y, sr = librosa.load(bass_stem_path, sr=44100)

 f0_clean = detect_pitch(y, sr)

In [ ]:
)import librosa
import numpy as np

class BassTabGenerator:
    def __init__(self):
        # 4현 베이스 표준 튜닝 (MIDI Note Number)
        # 4번줄(E1)=28, 3번줄(A1)=33, 2번줄(D2)=38, 1번줄(G2)=43
        self.tuning = [28, 33, 38, 43]
        self.string_names = ["E", "A", "D", "G"]

        # 분석된 노트 이벤트 저장소
        self.events = []

    def hz_to_fret(self, hz):
        """
        주파수(Hz)를 입력받아 (줄 인덱스, 프렛 번호)를 반환하는 핵심 로직
        """
        if hz is None or hz == 0 or np.isnan(hz):
            return None

        # 1. Hz -> MIDI 노트 번호 변환 (반올림)
        midi_note = int(round(librosa.hz_to_midi(hz)))

        # 2. 가능한 모든 운지 위치(Position) 찾기
        candidates = []
        for string_idx, open_note in enumerate(self.tuning):
            fret = midi_note - open_note
            # 베이스는 보통 0~24프렛
            if 0 <= fret <= 24:
                candidates.append((string_idx, fret))

        if not candidates:
            return None # 표현 불가능한 음

        # 3. [알고리즘] 운지 결정 로직 (Lowest Fret Priority)
        # 가장 낮은 프렛(개방현 쪽)을 우선 선택
        best_pos = min(candidates, key=lambda x: x[1])

        return best_pos

    def process_audio(self, y, sr, f0_array, hop_length=512):
        """
        오디오를 분석하여 온셋을 감지하고, 피치를 안정화하여 이벤트를 수집합니다.
        (출력은 하지 않고 데이터만 저장합니다)
        """
        # 기존 데이터 초기화
        self.events = []

        # 1. 온셋 감지
        onset_frames = librosa.onset.onset_detect(
            y=y, sr=sr, hop_length=hop_length,
            backtrack=True, units='frames'
        )

        # 2. 각 온셋 지점 분석
        for onset_frame in onset_frames:
            # 피치 안정화 (Attack 이후 5프레임 뒤의 Median 값 사용)
            check_window = 5

            # 배열 범위 체크
            start_idx = onset_frame
            end_idx = min(onset_frame + check_window, len(f0_array))

            # 유효한 피치 후보 수집
            pitch_candidates = [
                f0_array[i] for i in range(start_idx, end_idx)
                if not np.isnan(f0_array[i])
            ]

            if not pitch_candidates:
                continue

            # 중간값으로 대표 피치 결정
            stable_pitch = np.median(pitch_candidates)

            # 프렛 위치 변환
            pos = self.hz_to_fret(stable_pitch)

            if pos:
                current_time = librosa.frames_to_time(onset_frame, sr=sr, hop_length=hop_length)

                # 이벤트 저장
                self.events.append({
                    'time': current_time,
                    'string_idx': pos[0], # 0=E, 3=G
                    'fret': pos[1]
                })

    def display_tab(self, chars_per_line=80):
        """
        수집된 self.events 데이터를 바탕으로 가로형 타브 악보를 출력합니다.
        chars_per_line: 한 줄에 표시할 최대 문자 수 (자동 줄바꿈)
        """
        print(f"\nGenerated Horizontal Bass Tab")
        print("Standard Tuning (G-D-A-E)\n")

        # 각 줄의 문자열 버퍼 (0:G, 1:D, 2:A, 3:E)
        # 타브 악보는 G가 맨 위이므로 인덱싱 주의
        line_buffers = ["G |", "D |", "A |", "E |"]

        last_time = 0.0

        for event in self.events:
            string_idx = event['string_idx'] # 0(E) ~ 3(G)
            fret = event['fret']

            # --- 리듬 간격 조절 (Rhythmic Spacing) ---
            # 이전 음과의 시간 차이에 비례하여 대시(-) 추가
            time_diff = event['time'] - last_time
            # 0.1초당 대시 1개 (최소 1개)
            num_dashes = max(1, int(time_diff * 10))
            spacer = "-" * num_dashes

            # --- 4줄 채우기 ---
            for i in range(4):
                # line_buffers[0]은 G현(string_idx 3)
                # line_buffers[3]은 E현(string_idx 0)
                current_string_target = 3 - i

                if current_string_target == string_idx:
                    # 해당 줄에 프렛 번호 추가
                    line_buffers[i] += spacer + str(fret)
                else:
                    # 다른 줄에는 대시만 추가 (프렛 번호 자릿수만큼 공백 확보)
                    pad_len = len(str(fret))
                    line_buffers[i] += spacer + ("-" * pad_len)

            last_time = event['time']

            # --- 줄 바꿈 (Wrapping) ---
            if len(line_buffers[0]) > chars_per_line:
                self._print_system(line_buffers)
                # 버퍼 초기화 (헤더 재삽입)
                line_buffers = ["G |", "D |", "A |", "E |"]

        # 남은 내용이 있으면 출력
        if len(line_buffers[0]) > 3:
            self._print_system(line_buffers)

    def _print_system(self, buffers):
        """4줄 버퍼를 출력하는 내부 함수"""
        for line in buffers:
            print(line + "-|")
        print("") # 빈 줄

# ==========================================
# 실행 코드
# ==========================================

# 1. 인스턴스 생성
generator = BassTabGenerator()

# 2. 오디오 분석 (데이터 수집)
# y, sr, f0_clean은 이전 단계 피치 트래킹 결과 변수
generator.process_audio(y, sr, f0_clean, hop_length=512)

# 3. 악보 출력 (가로 모드)
generator.display_tab(chars_per_line=80)


Generated Horizontal Bass Tab
Standard Tuning (G-D-A-E)

G |-----------------------------------------------------------1----------------------|
D |--------------------------------1--2----------4---0-----1-----------1-----------1-|
A |---------------1---------1-1-3----------------------------------------------------|
E |-4----------4------4--4------------------4--------------------4--4-----4----------|

G |---------------------------------------------------------------------------1----|
D |------------------------------------------------------------------------3-------|
A |-4----0--1-----1--1-------0-0--1--1-----1-------1---1-----4--------4----------4-|
E |---4-------------------4-------------------------------4------------------------|

G |-----1---1--------------------------------------6--6-5----4--1------------------|
D |--2---------4-------------------------------4-----------------------------------|
A |------------------4---4-4---3-4----4-------------------------------------------